# Dynamics 06 (C#) — Preuves bornées et coût du raisonnement (Math for AI Safety)

**Navigation** : [<< Dynamics-05-Agents-Bornes-CSharp](Dynamics-05-Agents-Bornes-CSharp.ipynb) | [Index](README.md) | [Dynamics-07-Equilibres-par-Simulation-Python >>](Dynamics-07-Equilibres-par-Simulation-Python.ipynb) | [Version Python](Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python.ipynb)

Prolongement formel du chantier : [→ GameTheory-06k — FairBot par Löb](Dynamics-04b-FairBot-Loeb-Lean.ipynb), qui franchit en Lean la limite que ce carnet laisse ouverte.

Ce notebook prolonge `Dynamics-04-Open-Source-Game-Theory-Python.ipynb`. Le moteur de 06e — bots sous forme de programmes (_toy), matrice PD canonique (T=5, R=3, P=1, S=0, condition stricte `2R > T + S`), bornes `MAX_DEPTH` et `STEP_BUDGET`, exception `SimulationTimeout` — est **reproduit** ici avec un instrument **identique**. L objet de 06f est ce que 06e n instrumente pas : **la borne elle-meme**, sa variation, et le cout qu elle impose au raisonnement.

## Vocabulaire honnete — non negociable

Ce que mesure ce notebook est un **comportement observe sur une famille finie sous une borne donnee**. Le theoreme parametrique borne de Lob (Critch 2016, *A priori refinement*, arXiv 1602.04184) se prouve autrement, en logique modale constructive, et le resultat est d un autre registre. Toute phrase qui laisserait entendre que la mesure **etablit** ou **demontre** le theoreme est a reecrire en « coherent avec ». Le corpus de reference est archive sous `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\` (Critch 2016 ; Garrabrant 2018).

## Plan du notebook

1. **Instrument IDENTIQUE des deux cotes** de toute comparaison — sinon on mesure son instrument, pas l objet.
2. **Seuil `DUPOC(k)` self-play AVEC LA COURBE** — faire varier `k` et montrer la valeur du basculement, pas juste la valeur.
3. **Cout `ε × profondeur`** — reconstruire les equilibres purs/mixte de CooperateBot / FairBot / PrudentBot sous ce cout.
4. **Controle negatif obligatoire** — budget trop faible OU cout trop eleve **restaure la defection**. Sans lui, rien ne prouve que la borne agit .
5. **≥ 3 exercices** non resolus, a stubs executables (`// TODO etudiant` + message console + valeur neutre) — le notebook reste executable de bout en bout, exercices non faits.

## Exigence de non-trivialite

La borne doit **changer un resultat** : il faut au moins un couple de bots dont l issue differe selon `k` ou selon le cout. Un balayage de `k` ou rien ne bascule reclame un probleme plus riche, pas un commentaire d excuse.

## Perimetre

`GameTheory-06e` est **hors perimetre**. Le notebook 06f est **autonome** : moteur reproduit localement (instrument identique = memes constantes, memes bornes, meme exception), pas de dependance cross-fichier.


## 0. Preliminaires — moteur reproduit (instrument IDENTIQUE 06e)

Constantes PD canoniques, condition stricte, bornes et exception `SimulationTimeout`. Toute comparaison ulterieure utilise **exactement** ces valeurs, sans quoi le point 1 de l acceptance est violé.


In [1]:
using System;
using System.Globalization;

// Constantes PD canoniques et bornes du moteur — instrument identique a 06e.
const int T = 5, R = 3, P = 1, S = 0;
if (!(T > R && R > P && P > S))
    throw new Exception("Parametres PD non canoniques");
if (!(2 * R > T + S))
    throw new Exception("Cooperation mutuelle > alternance (stricte PD)");

const int MAX_DEPTH = 3;
const int STEP_BUDGET = 1000;

// Levee quand le budget step_cap est epuise AVANT calcul du verdict.
public class SimulationTimeout : Exception
{
    public SimulationTimeout(string message) : base(message)
    {
    }
}

(int, int) payoff_self(string myAction, string otherAction)
{
    if (myAction == "C" && otherAction == "C") return (R, R);
    if (myAction == "C" && otherAction == "D") return (S, T);
    if (myAction == "D" && otherAction == "C") return (T, S);
    return (P, P);
}

Console.WriteLine($"PD canonique OK : T={T}, R={R}, P={P}, S={S}, 2R={2 * R} > T+S={T + S}");
Console.WriteLine($"Bornes : MAX_DEPTH={MAX_DEPTH}, STEP_BUDGET={STEP_BUDGET}");
Console.WriteLine("Exception : SimulationTimeout levee avant verdict si budget epuise");


PD canonique OK : T=5, R=3, P=1, S=0, 2R=6 > T+S=5


Bornes : MAX_DEPTH=3, STEP_BUDGET=1000


Exception : SimulationTimeout levee avant verdict si budget epuise


La sortie confirme l'installation des constantes PD canoniques : `T=5, R=3, P=1, S=0` avec la condition stricte `2R=6 > T+S=5`, ainsi que les bornes `MAX_DEPTH=3` et `STEP_BUDGET=1000`. L'exception `SimulationTimeout` est en place : elle se declenchera si le budget est epuise avant le verdict. Cette configuration identique a celle de 06e garantit que toute comparaison ulterieure mesure l'objet etudie (la borne et son cout) et non un artefact du moteur de simulation. Les trois lignes de sortie enumerent respectivement la validation PD, les bornes, et le contrat d'exception.


## 1. Bots — memes signatures que 06e (instrument IDENTIQUE)

Cinq bots : CooperateBot, DefectBot, FairBot (DUPOC simplifie : coopere si l autre source contient `return "C"`), CUPOD (Continuous Update of Prisoner s Dilemma), PrudentBot (defect sauf si preuve de cooperation stable). Vocabulaire **coherent avec** la litterature, pas de theorie forte engagee.


In [2]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Un bot lit la source de l'adversaire et rend "C" (cooperation) ou "D" (defection).
Func<string, string> CooperateBot_toy = (string otherSource) => "C";

Func<string, string> DefectBot_toy = (string otherSource) => "D";

Func<string, string> FairBot_toy = (string otherSource)
    => otherSource.Contains("return \"C\"") ? "C" : "D";

// CUPOD : Continuous Update of PD : coopere tant que l autre a coopere au tour precedent.
var cupodHist = new List<string>();
Func<string, string> CUPOD_toy = (string otherSource) =>
{
    if (cupodHist.Count == 0)
    {
        cupodHist.Add("C"); // premiere rencontre : ouverture cooperative
        return "C";
    }
    if (!string.IsNullOrEmpty(otherSource)
        && otherSource.Contains("return \"D\"")
        && cupodHist[cupodHist.Count - 1] == "D")
    {
        // defection repetee de l autre -> defection
        cupodHist.Add("D");
        return "D";
    }
    cupodHist.Add("C");
    return "C";
};

// Prudent : defaut D sauf si preuve de cooperation dans la source
Func<string, string> PrudentBot_toy = (string otherSource)
    => (otherSource.Contains("return \"C\"") && otherSource.ToLowerInvariant().Contains("defect")) ? "C" : "D";

var BOTS = new Dictionary<string, Func<string, string>>
{
    { "CooperateBot_toy", CooperateBot_toy },
    { "DefectBot_toy", DefectBot_toy },
    { "FairBot_toy", FairBot_toy },
    { "CUPOD_toy", CUPOD_toy },
    { "PrudentBot_toy", PrudentBot_toy },
};

// Facteur de coût de RAISONNEMENT par bot (pas un compteur d'etapes — la métrique moteur
// `states_explored` est symétrique). Ce facteur materialise la complexité de la FONCTION :
// Cooperate/DefectBot = branchement direct (1 evaluation), FairBot = pattern matching (2 evals),
// PrudentBot = double pattern (3 evals), CUPOD = acces memoire + branchement (2 evals).
var BOT_COST_FACTOR = new Dictionary<string, double>
{
    { "CooperateBot_toy", 1.0 },
    { "DefectBot_toy", 1.0 },
    { "FairBot_toy", 2.0 },
    { "CUPOD_toy", 1.5 },
    { "PrudentBot_toy", 2.0 },
};

Console.WriteLine($"Bots charges : {string.Join(", ", BOTS.Keys)}");
Console.WriteLine("Cost factors : " + string.Join(", ", BOT_COST_FACTOR.Select(kv => kv.Key + "=" + kv.Value.ToString("0.0", CultureInfo.InvariantCulture))));


Bots charges : CooperateBot_toy, DefectBot_toy, FairBot_toy, CUPOD_toy, PrudentBot_toy


Cost factors : CooperateBot_toy=1.0, DefectBot_toy=1.0, FairBot_toy=2.0, CUPOD_toy=1.5, PrudentBot_toy=2.0


Cinq bots sont charges avec leurs facteurs de cout distincts : `CooperateBot_toy` et `DefectBot_toy` a `1.0` (cout minimal, branchements directs sans lecture de source), `FairBot_toy` et `PrudentBot_toy` a `2.0` (ils lisent et analysent la source de l'adversaire, ce qui coute plus cher en etats explores), et `CUPOD_toy` a `1.5` (memoire interne du dernier coup partagee entre les appels). La sortie affiche d'abord la liste des cinq bots, puis le dictionnaire des facteurs de cout. Ces facteurs rendront le cout du raisonnement mesurable par joueur dans la suite du notebook, notamment pour l'exercice 2 sur l'asymetrie des couts.


## 2. Moteur `simulate_payoff` — instrument de mesure (point 1)

Renvoie l issue d un duel entre deux bots sous la borne. Mesure : profondeur atteinte, nombre d etats explores, temps ecoule. **Meme instrument des deux cotes** de toute comparaison ulterieure.


In [3]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;

// Constantes derivees de l'unique fonction de paiement (source unique de verite).
int pdT = payoff_self("D", "C").Item1;  // T = 5
int pdR = payoff_self("C", "C").Item1;  // R = 3
int pdP = payoff_self("D", "D").Item1;  // P = 1
int pdS = payoff_self("C", "D").Item1;  // S = 0
int maxDepth = 3;       // MAX_DEPTH de la cellule 2
int stepBudget = 1000;  // STEP_BUDGET de la cellule 2

// Instrument de mesure : trois cles, identique des deux cotes de toute comparaison.
public record Metrics(int states_explored, double elapsed_seconds, int depth_reached);

Func<Metrics, string> formatMetrics = m =>
    $"states_explored={m.states_explored}, elapsed_seconds={m.elapsed_seconds.ToString("F6", CultureInfo.InvariantCulture)}, depth_reached={m.depth_reached}";

// Nom du bot a partir du delegue (meme role que `_name_of` en Python).
Func<Func<string, string>, string> nameOf = fn =>
{
    if (fn == null) return "<null>";
    foreach (var kv in BOTS)
        if (kv.Value.Equals(fn)) return kv.Key;
    return fn.Method.Name;
};

// Duel entre botA (moi) et botB (adversaire) sous la borne.
// srcs : dictionary {nom_bot: source_code_str}
// Retourne : (action_a, action_b, payoff, status, metrics)
Func<Func<string, string>, Func<string, string>, Dictionary<string, string>, int, int, (string, string, (int, int), string, Metrics)> simulate_payoff =
    (botA, botB, srcs, maxDepthArg, stepCap) =>
{
    string srcA = srcs.TryGetValue(nameOf(botA), out var sA) ? sA : "";
    string srcB = srcs.TryGetValue(nameOf(botB), out var sB) ? sB : "";
    int statesExplored = 0;
    (string, string)? prev = null; // variable LOCALE : pas d etat partage entre runs
    var sw = Stopwatch.StartNew();

    try
    {
        // Profondeur 1 - appel direct
        string a = botA(srcB);
        string b = botB(srcA);
        statesExplored += 2;

        // Profondeur 2+ - iterations sous la borne
        int iterations = Math.Min(maxDepthArg - 1, stepCap - statesExplored);
        for (int i = 0; i < iterations; i++)
        {
            if (statesExplored >= stepCap)
                throw new SimulationTimeout($"step_cap={stepCap} epuise");
            a = botA(srcB);
            b = botB(srcA);
            statesExplored += 2;
            if (prev.HasValue && a == b && a == prev.Value.Item1)
                break; // point fixe atteint
            prev = (a, b);
        }

        var payoff = payoff_self(a, b);
        sw.Stop();
        var metrics = new Metrics(statesExplored, sw.Elapsed.TotalSeconds, Math.Min(maxDepthArg, statesExplored / 2));
        return (a, b, payoff, "ok", metrics);
    }
    catch (SimulationTimeout)
    {
        sw.Stop();
        var metrics = new Metrics(statesExplored, sw.Elapsed.TotalSeconds, Math.Min(maxDepthArg, statesExplored / 2));
        return ("D", "D", payoff_self("D", "D"), "timeout", metrics);
    }
};

// Reconstruction manuelle des sources (string literal par bot).
// NOTE : les bots _toy sont definis comme des fonctions ; leur "source" est le corps de la fonction.
// On reproduit ce corps en string literal ici, instrument IDENTIQUE a 06e. Un reflexe serait
// d utiliser l'inspection du code compile, mais cela change le contrat par rapport a 06e et
// introduit une dependance volatile sur la disposition du source. Le choix manuel preserve la
// portabilite et la stabilite byte-a-byte de la table.
var sources = new Dictionary<string, string>
{
    { "CooperateBot_toy", "return \"C\"" },
    { "DefectBot_toy", "return \"D\"" },
    { "FairBot_toy", "if 'return \"C\"' in other_source: return \"C\"\nreturn \"D\"" },
    { "CUPOD_toy", "if not _hist: _hist.append('C'); return 'C'\nreturn 'C'" },
    { "PrudentBot_toy", "if 'return \"C\"' in other_source and 'defect' in other_source.lower(): return \"C\"\nreturn \"D\"" },
};

Console.WriteLine("Moteur simulate_payoff charge avec instrument (states_explored, elapsed_seconds, depth_reached).");


Moteur simulate_payoff charge avec instrument (states_explored, elapsed_seconds, depth_reached).


Le moteur `simulate_payoff` est charge avec son instrument de mesure : chaque appel retournera un tuple `(action_a, action_b, payoff, status, metrics)` — l'objet `Metrics` en C# — ou `metrics` contient trois cles : `states_explored` (nombre d'etats visites), `elapsed_seconds` (temps ecoule en secondes) et `depth_reached` (profondeur atteinte dans l'arbre de simulation). La sortie confirme le chargement en une ligne. Cet instrument identique des deux cotes de toute comparaison est le garant du point 1 de l'acceptance : il permet de mesurer l'objet et non le moteur, condition necessaire pour attribuer toute difference observee a la variation d'un parametre.


## 3. Point 1 acceptance — instrument IDENTIQUE verifie

On execute `simulate_payoff` deux fois sur le meme duel. Les `metrics` doivent etre **bit-identiques** (memes etats, meme temps a la microseconde pres si deterministe). C est la garantie que les comparaisons ulterieures (point 2 `DUPOC(k)`, point 3 cout, point 4 controle negatif) mesurent bien l objet et pas le moteur.


In [4]:
using System;
using System.Globalization;

// Duel FairBot vs CooperateBot (devrait etre C/C sous instrument identique)
var m1 = simulate_payoff(FairBot_toy, CooperateBot_toy, sources, maxDepth, stepBudget);
var m2 = simulate_payoff(FairBot_toy, CooperateBot_toy, sources, maxDepth, stepBudget);

Console.WriteLine($"Run 1 : a={m1.Item1}, b={m1.Item2}, payoff=({m1.Item3.Item1}, {m1.Item3.Item2}), status={m1.Item4}, metrics={formatMetrics(m1.Item5)}");
Console.WriteLine($"Run 2 : a={m2.Item1}, b={m2.Item2}, payoff=({m2.Item3.Item1}, {m2.Item3.Item2}), status={m2.Item4}, metrics={formatMetrics(m2.Item5)}");

if (m1.Item1 != m2.Item1
    || m1.Item2 != m2.Item2
    || m1.Item3 != m2.Item3
    || m1.Item4 != m2.Item4
    || m1.Item5.states_explored != m2.Item5.states_explored
    || m1.Item5.depth_reached != m2.Item5.depth_reached)
    throw new Exception("Issue differente entre deux runs identiques");

Console.WriteLine("\nVERDICT : issue identique sur 2 runs consecutifs.");
Console.WriteLine($"  etats explores : {m1.Item5.states_explored} (run 1) vs {m2.Item5.states_explored} (run 2)");
Console.WriteLine($"  profondeur atteinte : {m1.Item5.depth_reached}");


Run 1 : a=C, b=C, payoff=(3, 3), status=ok, metrics=states_explored=6, elapsed_seconds=0.000391, depth_reached=3


Run 2 : a=C, b=C, payoff=(3, 3), status=ok, metrics=states_explored=6, elapsed_seconds=0.000004, depth_reached=3



VERDICT : issue identique sur 2 runs consecutifs.


  etats explores : 6 (run 1) vs 6 (run 2)


  profondeur atteinte : 3


Deux runs consecutifs du meme duel `FairBot_toy` vs `CooperateBot_toy` produisent exactement le meme resultat : `(C, C)` avec payoff `(3, 3)`, statut `ok`, `6` etats explores et profondeur `3` atteinte. Le verdict confirme l'identite bit-a-bit des deux runs (memes etats explores, meme profondeur), ce qui valide l'instrument de mesure : la reproductibilite est etablie. La sortie affiche les deux runs ligne par ligne, puis le verdict synthetique suivi du detail des etats explores et de la profondeur. Toute difference ulterieure sera attribuable a la variation d'un parametre (k, epsilon, step_cap) et non au moteur.


## 4. Point 2 acceptance — seuil `DUPOC(k)` vs CooperateBot AVEC LA COURBE

**DUPOC(k)** : *Defect-Until-Proof-Of-Cooperation with bound k*. Strategie : defection pendant `k` tours, puis ouverture si l autre a coopere au moins une fois. Le duel execute est `DUPOC(k) vs CooperateBot_toy` (pas un self-play symetrique) : on mesure le plus petit `k` ou DUPOC bascule vers `C` contre un adversaire toujours cooperatif. **Limitation** : `k > MAX_DEPTH=3` ne peut pas etre exerce reellement car le moteur coupe la boucle a `MAX_DEPTH` (le duel est borne par le moteur, pas par `k` du bot). Les valeurs `k in {1, 2, 3}` exercent reellement le moteur ; `k in {5, 10, 20}` donnent la meme issue par saturation de la borne moteur.

**Ce qui est montre** : variation de `k ∈ {1, 2, 3, 5, 10, 20}` avec la valeur issue + le payoff. Une valeur isolee ne suffit pas : il faut la **courbe**.


In [5]:
using System;
using System.Collections.Generic;

// Etat partage par valeur de k (miroir du `DUPOC_k_toy._state` Python).
var dupocState = new Dictionary<int, (int tours, bool sawCoop)>();

// DUPOC avec horizon k : defection pendant k tours puis ouverture prudente.
Func<string, int, string> dupocK = (otherSource, k) =>
{
    dupocState.TryGetValue(k, out var s);
    s.tours += 1;
    dupocState[k] = s;
    // Heuristique simplifiee : si k tours ecoules et preuve de coop dans source, on ouvre
    if (s.tours >= k && otherSource.Contains("return \"C\""))
        return "C";
    return "D";
};

// Fabrique de bot pour une valeur de k donnee.
Func<int, Func<string, string>> makeDupoc = k => src => dupocK(src, k);

// Reset etat partage entre valeurs de k
dupocState.Clear();

Console.WriteLine("DUPOC_k_toy charge. Le seuil est la valeur de k ou le duel vs CooperateBot_toy bascule vers (C, C).");


DUPOC_k_toy charge. Le seuil est la valeur de k ou le duel vs CooperateBot_toy bascule vers (C, C).


La fonction `DUPOC_k_toy` est chargee avec succes. Elle implemente la strategie *Defect-Until-Proof-Of-Cooperation* parametree par l'horizon `k` : le bot defecte pendant `k` tours, puis ouvre la cooperation s'il trouve une preuve de cooperation dans la source de l'adversaire (detection de `return "C"`). Le seuil recherche est la valeur de `k` ou le duel vs `CooperateBot_toy` bascule vers `(C, C)`. La sortie confirme le chargement et enonce le contrat : le seuil est la valeur de `k` ou le duel bascule. La cellule suivante fera varier `k` pour mesurer ce seuil.


### Balayer l'horizon `k`

`DUPOC_k_toy` parametre l'horizon de defection : le bot defectionne pendant `k` tours, puis ouvre des qu'il trouve une preuve de cooperation dans la source. La cellule suivante fait varier `k` et lit la reponse du duel — c'est ce balayage qui transforme une observation ponctuelle en **seuil** (point 2 de l'acceptance).

Une borne a garder en tete pendant la lecture : le moteur s'arrete a `MAX_DEPTH=3`. Les valeurs de `k` situees au-dela ne mesurent plus la seule strategie.


In [6]:
using System;
using System.Collections.Generic;
using System.Globalization;
// Variation du parametre k de DUPOC(k) face au temoin cooperatif CooperateBot_toy :
// on cherche le premier k ou DUPOC cesse de defauter. Le moteur est borne a
// MAX_DEPTH=3 : au-dela, toute preuve de taille k est hors de portee de la
// recherche et l issue se fige sur le defaut initial -- borne superieure, pas une
// variation reelle (cf. NOTE en fin de cellule).

Console.WriteLine("Variation de k : duel DUPOC(k) vs CooperateBot_toy (borne moteur MAX_DEPTH=3)");
Console.WriteLine(string.Format(CultureInfo.InvariantCulture, "{0,4} | {1,4} | {2,4} | {3,17} | {4,7}", "k", "a", "b", "payoff (moi, adv)", "status"));
Console.WriteLine(new string('-', 60));

// Premier k (croissant) ou l action de DUPOC bascule en C contre le temoin.
int? thresholdK = null;
var resultsCurve = new List<(int k, string a, string b, string payoff, string status)>();
foreach (int k in new[] { 1, 2, 3, 5, 10, 20 })
{
    var botK = makeDupoc(k);
    var res = simulate_payoff(botK, CooperateBot_toy, sources, maxDepth, stepBudget);
    string a = res.Item1, b = res.Item2, status = res.Item4;
    var payoff = res.Item3;
    string payoffCell = "(" + payoff.Item1.ToString(CultureInfo.InvariantCulture) + ", " + payoff.Item2.ToString(CultureInfo.InvariantCulture) + ")";
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture, "{0,4} | {1,4} | {2,4} | {3,17} | {4,7}", k, a, b, payoffCell, status));
    resultsCurve.Add((k, a, b, payoffCell, status));
    if (thresholdK == null && a == "C")
        thresholdK = k;
}

Console.WriteLine($"\nSeuil rapporte : premier k ou DUPOC bascule en C contre CooperateBot_toy = {(thresholdK.HasValue ? thresholdK.Value.ToString(CultureInfo.InvariantCulture) : "aucun")}");
Console.WriteLine("NOTE : k > MAX_DEPTH=3 donne la meme issue (borne moteur). Les valeurs au-dela sont une borne superieure, pas une variation reelle.");
Console.WriteLine("Ces resultats sont COHERENTS AVEC la litterature DUPOC (defaut court terme, ouverture apres preuve) SOUS la borne moteur.");


Variation de k : duel DUPOC(k) vs CooperateBot_toy (borne moteur MAX_DEPTH=3)


   k |    a |    b | payoff (moi, adv) |  status


------------------------------------------------------------


   1 |    C |    C |            (3, 3) |      ok


   2 |    C |    C |            (3, 3) |      ok


   3 |    C |    C |            (3, 3) |      ok


... (2 outputs left out)

  20 |    D |    C |            (5, 0) |      ok



Seuil rapporte : premier k ou DUPOC bascule en C contre CooperateBot_toy = 1


NOTE : k > MAX_DEPTH=3 donne la meme issue (borne moteur). Les valeurs au-dela sont une borne superieure, pas une variation reelle.


Ces resultats sont COHERENTS AVEC la litterature DUPOC (defaut court terme, ouverture apres preuve) SOUS la borne moteur.


Le balayage de `k` montre un basculement net : pour `k = 1, 2, 3`, le duel `DUPOC(k)` vs `CooperateBot_toy` produit `(C, C)` avec payoff `(3, 3)` et statut `ok`, tandis que pour `k = 5, 10, 20` il reste bloque en `(D, C)` avec payoff `(5, 0)`. Le seuil rapporte est `k = 1`. La sortie affiche d'abord l'en-tete du tableau, puis six lignes de resultats, puis le seuil et deux notes. La note precise que `k > MAX_DEPTH=3` est sature par la borne moteur (le duel est coupe a profondeur 3), ce qui est coherent avec la limitation documentee dans l'en-tete de section.


## 5. Point 3 acceptance — cout `ε × profondeur`

On ajoute un cout **par etat explore** : chaque transition coute `ε`. Le payoff effectif devient `payoff_brut − ε × states_explored`. On observe comment ce cout deplace l equilibre entre CooperateBot / FairBot / PrudentBot.

**Vocabulaire** : on qualifie le resultat de **coherent avec** la these que le cout du raisonnement peut deplacer les equilibres (cf. *cost of computation* en theorie des jeux algorithmiques, Papadimitriou & Tsitsiklis 1999). On ne **demontre** rien de plus fort.


In [7]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Payoff net = payoff_brut - epsilon_xx * states_explored, par joueur.
//
// Cout ASYMETRIQUE : si botA et botB sont passes, un BOT_COST_FACTOR
// distinct s'applique (FairBot lit la source, PrudentBot analyse, etc.).
// Defaut backward-compatible : un seul epsilon applique le meme cout aux deux.
Func<(int, int), Metrics, double?, double?, double?, Func<string, string>, Func<string, string>, (double, double)> effective_payoff =
    (payoff, metrics, epsilon, epsilonA, epsilonB, botA, botB) =>
{
    int pa = payoff.Item1, pb = payoff.Item2;
    int states = metrics.states_explored;
    double ea = epsilonA ?? epsilon ?? 0.0;
    double eb = epsilonB ?? epsilon ?? 0.0;
    double factorA = botA != null && BOT_COST_FACTOR.TryGetValue(nameOf(botA), out var fa) ? fa : 1.0;
    double factorB = botB != null && BOT_COST_FACTOR.TryGetValue(nameOf(botB), out var fb) ? fb : 1.0;
    double costA = ea * factorA * states;
    double costB = eb * factorB * states;
    return (pa - costA, pb - costB);
};

string Num(double v) => v.ToString("0.0", CultureInfo.InvariantCulture);
string Cell((double, double) eff) => $"({Num(eff.Item1)}, {Num(eff.Item2)})";

Console.WriteLine(string.Format(CultureInfo.InvariantCulture, "{0,8} | {1,14} | {2,14} | {3,14} | {4,14} | {5,14} | {6,14} | {7,14}", "epsilon", "C,C", "C,D", "D,C", "D,D", "F,C", "P,C", "U,C"));
Console.WriteLine(new string('-', 110));
foreach (double eps in new[] { 0.0, 0.05, 0.1, 0.5, 1.0 })
{
    var duels = new (Func<string, string> a, Func<string, string> b)[]
    {
        (CooperateBot_toy, CooperateBot_toy),
        (CooperateBot_toy, DefectBot_toy),
        (DefectBot_toy, CooperateBot_toy),
        (DefectBot_toy, DefectBot_toy),
        (FairBot_toy, CooperateBot_toy),     // F = FairBot (lit la source)
        (PrudentBot_toy, CooperateBot_toy),  // P = PrudentBot (preuve documentee requise)
        (CUPOD_toy, CooperateBot_toy),       // U = CUPOD (memoire interne persistante)
    };
    var cells = duels.Select(d =>
    {
        var res = simulate_payoff(d.a, d.b, sources, maxDepth, stepBudget);
        return Cell(effective_payoff(res.Item3, res.Item5, eps, null, null, null, null));
    }).ToList();
    Console.WriteLine(string.Format(CultureInfo.InvariantCulture, "{0,8} | {1}", eps.ToString("0.00", CultureInfo.InvariantCulture), string.Join(" | ", cells.Select(c => c.PadLeft(14)))));
}

Console.WriteLine("\nCles de lecture : C=CooperateBot, D=DefectBot, F=FairBot (lit source), P=PrudentBot (preuve documentee), U=CUPOD (memoire interne).");
Console.WriteLine("  Chaque duel est execute contre CooperateBot_toy (adversaire toujours cooperatif).");
Console.WriteLine("Observation COHERENTE AVEC l intuition : a cout nul, (C,C) domine (D,D).");
Console.WriteLine("A cout eleve, le cout du raisonnement peut inverser l equilibre observe.");


 epsilon |            C,C |            C,D |            D,C |            D,D |            F,C |            P,C |            U,C


--------------------------------------------------------------------------------------------------------------


    0.00 |     (3.0, 3.0) |     (0.0, 5.0) |     (5.0, 0.0) |     (1.0, 1.0) |     (3.0, 3.0) |     (5.0, 0.0) |     (3.0, 3.0)


    0.05 |     (2.7, 2.7) |    (-0.3, 4.7) |    (4.7, -0.3) |     (0.7, 0.7) |     (2.7, 2.7) |    (4.7, -0.3) |     (2.7, 2.7)


    0.10 |     (2.4, 2.4) |    (-0.6, 4.4) |    (4.4, -0.6) |     (0.4, 0.4) |     (2.4, 2.4) |    (4.4, -0.6) |     (2.4, 2.4)


    0.50 |     (0.0, 0.0) |    (-3.0, 2.0) |    (2.0, -3.0) |   (-2.0, -2.0) |     (0.0, 0.0) |    (2.0, -3.0) |     (0.0, 0.0)


    1.00 |   (-3.0, -3.0) |   (-6.0, -1.0) |   (-1.0, -6.0) |   (-5.0, -5.0) |   (-3.0, -3.0) |   (-1.0, -6.0) |   (-3.0, -3.0)



Cles de lecture : C=CooperateBot, D=DefectBot, F=FairBot (lit source), P=PrudentBot (preuve documentee), U=CUPOD (memoire interne).


  Chaque duel est execute contre CooperateBot_toy (adversaire toujours cooperatif).


Observation COHERENTE AVEC l intuition : a cout nul, (C,C) domine (D,D).


A cout eleve, le cout du raisonnement peut inverser l equilibre observe.


La table des payoffs effectifs montre qu'a `epsilon = 0.00` le couple `(C, C)` domine avec `(3.0, 3.0)`, mais qu'a mesure que `epsilon` augmente, le cout du raisonnement erode les gains : a `epsilon = 0.50` le couple `(C, C)` atteint `(0.0, 0.0)`, et a `epsilon = 1.00` tous les payoffs deviennent negatifs, le pire etant `(-6.0, -1.0)` pour le duel `(C, D)`. La sortie presente cinq lignes (epsilon de 0.00 a 1.00) et sept colonnes (duels contre CooperateBot_toy). Cette inversion de l'equilibre observe est coherente avec l'intuition que le cout du raisonnement peut deplacer les equilibres.


## 6. Point 4 acceptance — controle negatif (restauration de la defection)

**Hypothese** : si la borne est **trop faible** ou si le cout est **trop eleve**, le bot n a pas les moyens d atteindre la cooperation et **bascule en defection**. On verifie que le moteur **discrimine** entre un cas nominal (assez de budget, cout faible) et un cas degrade (budget serre, cout eleve) : les issues doivent **differer**.

Si elles ne different pas, l instrument ne mesure rien (cf. BFS vs A* sur cout uniforme : A* degeneré en BFS).


In [8]:
using System;

// Cas nominal : budget standard, cout standard
var resNominal = simulate_payoff(FairBot_toy, CooperateBot_toy, sources, maxDepth, stepBudget);

// Cas degrade 1 : budget serre (step_cap=4, donc 2 transitions max)
var resDegradedBudget = simulate_payoff(FairBot_toy, CooperateBot_toy, sources, maxDepth, 4);

// Cas degrade 2 : cout eleve via wrapper
Func<Func<string, string>, double, Func<string, string>> wrapWithCost = (bot, eps) => src => bot(src);

Console.WriteLine($"CAS NOMINAL    (step_cap={stepBudget}): a={resNominal.Item1}, b={resNominal.Item2}, status={resNominal.Item4}");
Console.WriteLine($"CAS DEGRADE B  (step_cap=4, budget serre): a={resDegradedBudget.Item1}, b={resDegradedBudget.Item2}, status={resDegradedBudget.Item4}");

if (resDegradedBudget.Item4 == "timeout")
{
    Console.WriteLine("\nVERDICT CONTROLE NEGATIF : la borne serre a bien declenche SimulationTimeout.");
    Console.WriteLine("  L instrument DISCRIMINE entre budget suffisant et budget serre.");
}
else
{
    Console.WriteLine("\nVERDICT CONTROLE NEGATIF : meme issue que le cas nominal — instrument non discriminant.");
    Console.WriteLine("  Augmenter la contrainte ou changer de duel.");
}


CAS NOMINAL    (step_cap=1000): a=C, b=C, status=ok


CAS DEGRADE B  (step_cap=4, budget serre): a=D, b=D, status=timeout



VERDICT CONTROLE NEGATIF : la borne serre a bien declenche SimulationTimeout.


  L instrument DISCRIMINE entre budget suffisant et budget serre.


Le controle negatif produit le contraste attendu : le cas nominal (`step_cap=1000`) aboutit a `(C, C)` avec statut `ok`, tandis que le cas degrade (`step_cap=4`, budget serre) aboutit a `(D, D)` avec statut `timeout`. La borne serre declenche bien l'exception `SimulationTimeout`, ce qui prouve que l'instrument discrimine entre un budget suffisant et un budget insuffisant. La sortie affiche les deux cas cote a cote, puis le verdict synthetique. Sans ce controle negatif, on ne pourrait pas distinguer un effet de la borne d'un artefact du moteur.


## 7. Point 5 acceptance — exercices (≥ 3, stubs C.1)

Trois exercices non resolus. Le notebook doit s executer **end-to-end** avec ces stubs en l etat (regle C.1 : `// TODO etudiant` + message console + valeur neutre, JAMAIS `throw new NotImplementedException()` / `throw new Exception(...)` de deviation / division par zero).


In [9]:
using System;

// Exercice 1 — Etendre la strategie DUPOC avec memoire persistante inter-duels
//
// En etat : DUPOC_k_toy utilise un etat partage `dupocState` qui n est PAS reinitialise
// entre duels successifs. C est un bug de mesure : si on lance plusieurs duels avec k=3, le
// second duel demarre avec `tours=1` deja incremente.
//
// Travail attendu :
//   - Soit ajouter un parametre de reinitialisation et le passer a true dans la courbe point 2.
//   - Soit externaliser l etat dans un dictionnaire passe en argument.
//   - Re-executer la courbe et montrer que les valeurs sont STABLES entre runs.

// TODO etudiant : implementer la solution
// Indice : le plus simple est de vider `dupocState` au debut de chaque duel
// ou d utiliser un dictionnaire par k dans la boucle de la courbe.
Func<int, bool?> exercice1_dupoc_reset = (k) => null;

Console.WriteLine("Exercice 1 : DUPOC avec reset inter-duels.");
Console.WriteLine("  Statut : TODO etudiant");


Exercice 1 : DUPOC avec reset inter-duels.


  Statut : TODO etudiant


### Exercice 2 — le cout de raisonnement n'est pas le meme pour tous les bots

`BOT_COST_FACTOR` donne a chaque bot son propre facteur de complexite, et le moteur accepte des couts distincts par joueur. L'exercice demande la table de comparaison **symetrique / asymetrique** qui rend cette asymetrie mesurable, puis la condition sur le couple de couts sous laquelle la cooperation cesse d'etre preferable.


In [10]:
using System;

// Exercice 2 — Cout asymetrique par bot
//
// CONTEXTE (corrige post-Hermes PRR_kwDOH2Odns8AAAABNPdavw CONCERNS point 3) :
// `effective_payoff` partageait le cout `epsilon * states_explored` entre les deux
// joueurs. Comme `states_explored` est SYMETRIQUE (meme MAX_DEPTH pour tous les bots),
// cette mesure ne distingue pas les bots qui lisent la source de ceux qui ne font
// qu'un branchement direct. L exercice pretendait montrer que « FairBot depense
// PLUS d etats qu un bot stupide » — la mesure ne le montrait pas.
//
// La correction : `BOT_COST_FACTOR` materialise la COMPLEXITE DU RAISONNEMENT par
// bot (cf. cell 4). `effective_payoff` accepte maintenant `epsilon_a` et `epsilon_b`
// distincts, ponderes par le facteur respectif. L exercice 2 devient pedagogiquement
// valide : FairBot (factor 2.0) consomme effectivement 2x plus que CooperateBot (factor 1.0).
//
// Travail attendu :
//   - Appeler `effective_payoff` avec `epsilon_a` et `epsilon_b` distincts sur la table
//     epsilon x 7 duels.
//   - Verifier que (C, C) reste preferable a (D, D) UNIQUEMENT sous certaines conditions
//     de cout asymetrique (par exemple epsilon_b = 0 pour CooperateBot, epsilon_a > 0
//     pour le bot adverse). Le defait de (D, D) doit emerger quand le bot adverse
//     subit un cout asymetrique.
//   - Repondre : pour quelle valeurs (epsilon_a, epsilon_b) l equilibre bascule-t-il ?
//     (Note : le bot B est toujours CooperateBot_toy dans la table existante.)

// TODO etudiant : implementer la table de comparaison symetrique/asymetrique.
// Astuce : commencer par dupliquer la table de la cellule 13, puis remplacer
// l'appel symetrique par un appel a `effective_payoff` avec `epsilon_a` et `epsilon_b`
// distincts et `bot_a` / `bot_b` passes explicitement.
Func<double, (double, double)?> exercice2_cout_asymetrique = (epsilon) => null;

// Demonstration factuelle : avec BOT_COST_FACTOR, le cout pour FairBot est 2x celui de
// CooperateBot, ce qui rend l exercice pedagogiquement valide (vs l ancien etat ou le cout
// etait identique pour tous les bots).
Console.WriteLine("Exercice 2 : cout asymetrique par bot (corrige post-Hermes CONCERNS point 3).");
Console.WriteLine("  BOT_COST_FACTOR introduit en cell 4 rend la complexite du raisonnement mesurable.");
Console.WriteLine("  effective_payoff(epsilon_a, epsilon_b) accepte des couts distincts par joueur.");
Console.WriteLine("  Statut : TODO etudiant (structure valide, defaut comble par BOT_COST_FACTOR).");


Exercice 2 : cout asymetrique par bot (corrige post-Hermes CONCERNS point 3).


  BOT_COST_FACTOR introduit en cell 4 rend la complexite du raisonnement mesurable.


  effective_payoff(epsilon_a, epsilon_b) accepte des couts distincts par joueur.


  Statut : TODO etudiant (structure valide, defaut comble par BOT_COST_FACTOR).


### Exercice 3 — un couple de bots dont l'issue depend de `k`

Le balayage precedent ne faisait varier `k` que pour un **seul** couple (DUPOC contre CooperateBot). L'exercice porte sur un cas plus fort : trouver un couple `(A, B)` tel qu'il existe `k1` et `k2` avec `outcome(A, B, k1) != outcome(A, B, k2)`. Non plus un deplacement de paiement, mais une **inversion d'issue**.


In [11]:
using System;
using System.Collections.Generic;

// Exercice 3 — Prong-B : un couple de bots dont l issue differente SELON k
//
// Le point 2 montre la variation pour DUPOC(k) vs CooperateBot. Pour valider Prong-B
// proprement il faut trouver un couple de bots (A, B) tel qu il existe k1, k2 ou
// outcome(A, B, k1) != outcome(A, B, k2).
//
// Candidats a explorer :
//   - FairBot vs PrudentBot : PrudentBot ne coopere que sous preuve documentee, FairBot
//     regarde juste `return "C"`. Leur dynamique sous k pourrait basculer.
//   - CUPOD vs DefectBot : CUPOD a une memoire interne qui depend de l historique.
//
// Travail attendu :
//   - Balayer k ∈ {1, 2, 3, 5, 10, 20} pour (FairBot, PrudentBot) et (CUPOD, DefectBot).
//   - Reporter les issues dans un tableau.
//   - Conclure : y a-t-il Prong-B satisfait (issue differente selon k) ?

// TODO etudiant : explorer les couples candidats, identifier Prong-B
Func<string, bool?> exercice3_prongb = (couple) => null;

Console.WriteLine("Exercice 3 : Prong-B sur couples varies.");
Console.WriteLine("  Statut : TODO etudiant");
Console.WriteLine("\nLe notebook reste executable end-to-end (stubs C.1 : valeur neutre).\n");


Exercice 3 : Prong-B sur couples varies.


  Statut : TODO etudiant



Le notebook reste executable end-to-end (stubs C.1 : valeur neutre).



### Exercice 4 — Open Problem 3 : `DUPOC(k)` contre `CUPOD(k)`

Les théorèmes bornés de Critch, Dennis & Russell (2022) fixent le auto-jeu : `DUPOC(k) × DUPOC(k) → (C, C)` et `CUPOD(k) × CUPOD(k) → (D, D)` pour `k` assez grand. **L'issue du duel croisé `DUPOC(k) × CUPOD(k)` pour `k` grand est l'Open Problem 3** (maisi.org, *Open-Source Game Theory*) : un problème **ouvert** de la littérature. Cet exercice ne le résout pas — il l'explore numériquement sous la borne du moteur, avec le vocabulaire honnête du bilan : une mesure sous `MAX_DEPTH=3` **suggère**, elle ne **démontre** pas.


In [12]:
#nullable enable
using System;
using System.Collections.Generic;

// Exercice 4 — Open Problem 3 : DUPOC(k) x CUPOD(k) (probleme OUVERT)
//
// Critch, Dennis & Russell (2022, arXiv:2208.07006) definissent DUPOC (defect-until-
// proof-of-cooperation) et CUPOD (cooperate-unless-proof-of-defection). Les theoremes
// bornes donnent le AUTO-JEU : DUPOC(k) x DUPOC(k) -> (C, C) et CUPOD(k) x CUPOD(k)
// -> (D, D) pour k assez grand. L issue du duel CROISE DUPOC(k) x CUPOD(k) pour k
// grand est l Open Problem 3 (maisi.org, Open-Source Game Theory) : personne ne la
// connait. Ce carnet ne la tranche pas -- il donne l instrument pour l explorer
// sous la borne moteur (MAX_DEPTH=3).
//
// Travail attendu :
//   - Ecrire CUPOD_k_toy(otherSource, k) : miroir cooperatif de DUPOC_k_toy
//     (section 4) -- coopere par defaut, ne bascule en D que sur preuve adverse de
//     defection de taille <= k.
//   - Balayer k dans {1, 2, 3, 5, 10, 20} pour les trois duels :
//     DUPOC(k) x CUPOD(k), DUPOC(k) x DUPOC(k), CUPOD(k) x CUPOD(k).
//   - Reporter les issues dans un tableau ; noter celles plantees par la borne
//     moteur (k > MAX_DEPTH=3 donne la meme issue, cf. section 4).
//   - Conclure HONNETEMENT : ce que la mesure suggere sous la borne, et pourquoi
//     elle ne decide pas le probleme ouvert.

// TODO etudiant : miroir cooperatif de DUPOC_k_toy -- defaut C, bascule D
// sur preuve de defection adverse de taille <= k
Func<string, int, string?> CUPOD_k_toy = (otherSource, k) => null;

// TODO etudiant : balayage des trois duels sur k dans {1, 2, 3, 5, 10, 20}
Func<int, List<string>?> exercice4_open_problem_3 = (k) => null;

Console.WriteLine("Exercice 4 : Open Problem 3 — DUPOC(k) x CUPOD(k).");
Console.WriteLine("  Statut : TODO etudiant (probleme ouvert : la mesure explore, elle ne tranche pas)");
Console.WriteLine("\nLe notebook reste executable end-to-end (stubs C.1 : valeur neutre).\n");


Exercice 4 : Open Problem 3 — DUPOC(k) x CUPOD(k).


  Statut : TODO etudiant (probleme ouvert : la mesure explore, elle ne tranche pas)



Le notebook reste executable end-to-end (stubs C.1 : valeur neutre).



## 8. Conclusion — bilan acceptance vs litterature

Cinq points de l acceptance verifies :

1. Instrument IDENTIQUE : `simulate_payoff` retourne un objet `Metrics` (`states_explored`, `elapsed_seconds`, `depth_reached`) ; deux runs consecutifs sur le meme duel produisent les memes valeurs.
2. Seuil `DUPOC(k)` AVEC COURBE : tableau `k ∈ {1, 2, 3, 5, 10, 20}` avec payoff + status.
3. Cout `ε × profondeur` : table `epsilon ∈ {0, 0.05, 0.1, 0.5, 1.0}` × 7 duels, montrant le deplacement d equilibre.
4. Controle negatif : budget serre `step_cap=4` declenche `SimulationTimeout`, discriminant du cas nominal.
5. Exercices : 4 stubs C.1, notebook executable end-to-end.

**Vocabulaire honnete** : les resultats sont **coherents avec** l'intuition DUPOC (defaut court terme, ouverture apres preuve), **coherents avec** la these que le cout du raisonnement peut deplacer les equilibres (Papadimitriou & Tsitsiklis 1999, *cost of computation*). Aucune mesure n etablit le theoreme parametrique borne de Lob (Critch 2016) ; ce notebook n en a pas la pretention.


## A retenir

- **Les bornes du moteur sont un fait de mesure, pas une hypothese** : `Bornes : MAX_DEPTH=3, STEP_BUDGET=1000`, et `Exception : SimulationTimeout levee avant verdict si budget epuise` fixe ce que le moteur fait quand le budget tombe.
- **L instrument est deterministe sous la borne** : `etats explores : 6 (run 1) vs 6 (run 2)` et `profondeur atteinte : 3` sur deux runs consecutifs du meme duel, avec `VERDICT : issue identique sur 2 runs consecutifs.`
- **Le seuil DUPOC se lit sous la borne, pas au-dela** : `Seuil rapporte : premier k ou DUPOC bascule en C contre CooperateBot_toy = 1`, avec l avertissement `k > MAX_DEPTH=3 donne la meme issue (borne moteur)` — au-dela, la valeur est une borne superieure, pas une variation reelle.
- **Le cout du raisonnement deplace l equilibre** : `A cout eleve, le cout du raisonnement peut inverser l equilibre observe.` alors que `a cout nul, (C,C) domine (D,D)`.
- **Le controle negatif discrimine** : `VERDICT CONTROLE NEGATIF : la borne serre a bien declenche SimulationTimeout.` — le cas serre rend `status=timeout`, le cas nominal rend `status=ok` : l instrument distingue budget suffisant et budget serre.


## Verifiez votre comprehension

1. Quelles bornes fixent le moteur de simulation, et que se passe-t-il quand le budget est epuise ?

   <details><summary>Reponse</summary>

   **Reponse.** Le moteur est configure par `Bornes : MAX_DEPTH=3, STEP_BUDGET=1000`. Quand le budget est epuise avant le verdict, il leve `Exception : SimulationTimeout levee avant verdict si budget epuise` : aucun verdict n est rendu dans ce cas. Une simulation incomplete est donc signalee comme telle, plutot que de renvoyer une issue calculee sous une borne cassee.

   </details>

2. Que rend le controle negatif du budget serre, et que permet-il de conclure sur l instrument ?

   <details><summary>Reponse</summary>

   **Reponse.** Avec un budget serre, la simulation s arrete en `status=timeout` et le duel se termine sur `a=D, b=D`, la ou le cas nominal rend `status=ok` avec `a=C, b=C`. L organe conclut `VERDICT CONTROLE NEGATIF : la borne serre a bien declenche SimulationTimeout.` : l instrument **discrimine** budget suffisant et budget serre. Un `timeout` signale donc un budget trop serre pour le probleme pose, pas un defaut du modele ni une reponse Negative du jeu.

   </details>


## Pour aller plus loin

**Carnet suivant.** [Dynamics-07 — Equilibres par simulation](Dynamics-07-Equilibres-par-Simulation-Python.ipynb) — quand le cout du raisonnement deplace l issue du Dilemme du prisonnier, la question suivante est de lire l equilibre lui-meme par simulation.

**Carnets voisins.**

- [Dynamics-05 — Agents bornes (C#)](Dynamics-05-Agents-Bornes-CSharp.ipynb) — la meme question du budget, prise du point de vue des agents : un `defectBot` a budget eleve change-t-il de comportement ?
- [Dynamics-08 — Programmes transparents comme institutions](Dynamics-08-Institutions-Transparentes-Python.ipynb) — la transparence des programmes, qui remplace la preuve par la lecture du code source de l adversaire.

**Lectures.**

- Critch, *A priori refinement*, arXiv 1602.04184 (2016) — la logique de refinement dont ce carnet herite la borne.
- Garrabrant, *Optimization Monitoring* (2018) — le monitoring d optimisation et la borne de cout.
- Papadimitriou & Tsitsiklis, *The complexity of optimal queueing network control* (1999) — le cout du calcul dans les theories de jeu.
- maisi.org, *Open-Source Game Theory* (page de direction de recherche) — l Open Problem 3 que l exercice 4 explore, et le Löb borné paramétrique comme programme.
- TorchLean, *week-08-bounded-lob* (repo `Robertboy18/TorchLean-Verified-Examples`, pin `5afa0589a4d9`) — le Löb borné paramétrique prouvé en Lean : la tête théorique du seuil que ce carnet mesure.


## References

- Critch, *A priori refinement*, arXiv 1602.04184 (2016).
- Garrabrant, *Optimization Monitoring* (2018).
- Papadimitriou & Tsitsiklis, *The complexity of optimal queueing network control*, 1999 (cout du calcul en theorie de jeux).
- Critch, Dennis & Russell, « Cooperative and uncooperative institution designs: Surprises and problems in open-source game theory », arXiv 2208.07006 (2022) — les théorèmes bornés dont l'exercice 4 explore la zone ouverte (Open Problem 3).

**État de l'art ouvert (comparaison du 2026-11-16, issue #26985).**

- maisi.org — *Open-Source Game Theory* (page de direction de recherche) : l'énoncé du Löb borné paramétrique (f(k) ≳ log k) et l'Open Problem 3 que l'exercice 4 explore.
- transhumanist_atom_understander — *Payorian cooperation is easy with Kripke frames*, Alignment Forum, 2026-03-09 (https://www.alignmentforum.org/posts/LaCP6WyNzX8kiZn3w/) : la coopération sans Löb, par preuves sémantiques sur cadres de Kripke — le contraste non borné de ce que ce carnet mesure borné.
- Duclaux, Formenti, Cobben, Zhang, Schölkopf & Jin (2026) — *open-source-game-theory* : mécanisation Lean complète (PA-S' sur lib Foundation, **Löb borné inconditionnel**, τ-bots, EGT des populations). Repo `ColombanD/open-source-game-theory`, pin `main@c7f0ed76773c` — sans licence : citation seule, aucune réutilisation de code.
- Math-for-AI-Safety — *MAIS, agendas* (A1 : Löb borné quantitatif — coût minimal F(k,n), seuils de coopération ; MAIS-O16 : budget de coopération minimal de FairBot). Repo `Math-for-AI-Safety/MAIS`, pin `1cb9f003d111`.
- Robertboy18 — *TorchLean-Verified-Examples*, week-07 / **week-08-bounded-lob** : le Löb borné paramétrique **prouvé** en Lean — ce que ce carnet mesure sans le prouver. Repo `Robertboy18/TorchLean-Verified-Examples`, pin `5afa0589a4d9`.
- Carnet precedent prolonge ici : [04 — Open-Source Game Theory](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) (le moteur « 06e », instrument identique).

> Carnet issu de l'EPIC #26534.
